# PriceFM_Plus: reproduce and extend the results

This notebook shows how to install the package, download public artifacts, inspect one saved forecast, reproduce the paper's metrics and figures, rerun inference, extend ENTSO-E data, and train a custom model. Large downloads and training are **opt-in**. The released data live in [European_Foundation_Model_DAM](https://huggingface.co/datasets/RunyaoYu/European_Foundation_Model_DAM).

Use Python 3.10 or newer. Install the current release in a terminal or a fresh notebook environment:

```bash
python -m pip install "https://huggingface.co/datasets/RunyaoYu/European_Foundation_Model_DAM/resolve/main/pricefm_plus-0.1.0-py3-none-any.whl"
```

If you cloned this repository, `python -m pip install -e .` installs the same source. `pip install PriceFM_Plus` by name becomes available only after publication to a Python package index. Restart the kernel after installation if needed.


## 1. Choose an output directory

All API downloads and generated results use the current working directory. The example keeps them outside the cloned repository.


In [ ]:
from pathlib import Path
import os

workdir = Path.home() / "PriceFM_Plus_demo"
workdir.mkdir(parents=True, exist_ok=True)
os.chdir(workdir)
print("Outputs:", Path.cwd())

import PriceFM_Plus
print("Public functions:", PriceFM_Plus.__all__)


## 2. Download data, weights, and one saved forecast

Set `RUN_DOWNLOADS` to `True` to fetch the archives. `download_data("raw")` creates raw data under `Data/`; `download_data("processed")` creates the model-ready files under the same folder. Weights and forecasts go to `Weights/` and `Forecasts/`. The archives may take time and disk space. The first example downloads only one forecast family for inspection.


In [ ]:
RUN_DOWNLOADS = False  # Change to True when ready.

if RUN_DOWNLOADS:
    PriceFM_Plus.download_data("processed")
    # PriceFM_Plus.download_data("raw")  # Needed to extend raw records later.
    PriceFM_Plus.download_weights("PriceFM_Plus")
    PriceFM_Plus.download_forecasts("PriceFM_Plus")


In [ ]:
# Saved forecast NPZs contain dates, region labels, row indices, time slots,
# and three native-price quantile predictions. They contain no training logs.
import numpy as np

files = sorted(Path("Forecasts/PriceFM_Plus").rglob("fold1.npz"))
if files:
    with np.load(files[0], allow_pickle=False) as saved:
        print("Example:", files[0])
        print({name: saved[name].shape for name in saved.files})
else:
    print("Download forecasts above to inspect a saved file.")


## 3. Reproduce Table 3 from saved forecasts

This step does **not** load model weights or run inference. It scores saved predictions against observed prices in the processed data. Download all eight model forecast archives plus `Persistence`, then call `reproduce_metrics()`. The result is `Metrics/table3.csv`, with pinball loss, interval score, MAE, and RMSE for the three feature cases. It uses test folds 1–3; fold 4 is validation only.


In [ ]:
RUN_FULL_TABLE = False  # Downloads all forecast families when True.

if RUN_FULL_TABLE:
    PriceFM_Plus.download_data("processed")
    for model in (
        "Persistence", "MLP", "LSTM", "Transformer", "PriceFM_Plus",
        "TabPFN_TS_3", "TiRex_2", "TimesFM_3", "Chronos_2",
    ):
        PriceFM_Plus.download_forecasts(model)
    table3 = PriceFM_Plus.reproduce_metrics()
    display(table3.head())
    print("Saved:", Path("Metrics/table3.csv").resolve())


## 4. Regenerate the paper figures

Install the optional plotting dependencies from this clone with `python -m pip install -e ".[figures]"`. With the released wheel, use `python -m pip install "PriceFM_Plus[figures] @ https://huggingface.co/datasets/RunyaoYu/European_Foundation_Model_DAM/resolve/main/pricefm_plus-0.1.0-py3-none-any.whl"`. The function downloads released plotting source and **derived figure-input tables**, then regenerates five grouped PDFs and four ablation panels in `Figures/`. It does not derive those figure-input tables anew from the raw data or saved forecast archives.


In [ ]:
RUN_FIGURES = False

if RUN_FIGURES:
    figure_dir = PriceFM_Plus.reproduce_figures()
    print("Created:", *sorted(figure_dir.glob("*.pdf")), sep="\n  ")


## 5. Reproduce a forecast from archived weights

Install the `train` extra (`python -m pip install -e ".[train]"` from this clone). Download processed data and the trained model's weights. The example below runs one region, one feature case, and one test fold. Reproduction writes to `Forecasts/`. If you completed Section 3, use a fresh output directory before running this cell so the archived forecast is preserved.

The same call accepts `MLP`, `LSTM`, or `Transformer`. Full reproduction covers three feature cases and held-out folds 1–3 by default. For a foundation model, install its model-specific extra; if the upstream checkpoint is absent, the package asks before downloading it and requests an access token only when required.


In [ ]:
RUN_INFERENCE = False  # May use substantial memory; a GPU is recommended.

if RUN_INFERENCE:
    saved = Path("Forecasts/PriceFM_Plus/Realistic/Joint/L2/fold1.npz")
    if saved.exists():
        raise FileExistsError(f"Saved forecast already exists: {saved}. Use a fresh output directory.")
    PriceFM_Plus.download_data("processed")
    PriceFM_Plus.download_weights("PriceFM_Plus")
    paths = PriceFM_Plus.reproduce_forecasts(
        "PriceFM_Plus", regions=["AT"], cases=["Realistic"],
        folds=(1,), batch_days=1,
    )
    print("Generated:", *paths, sep="\n  ")


## 6. Add new ENTSO-E data

Download raw **and** processed data first. Supply your own ENTSO-E API token through the `ENTSOE_API_TOKEN` environment variable. The API extends the local raw records and the three processed fill variants for the selected dates; end dates are exclusive. Availability varies by region and feature. Never save a token in this notebook or commit it to Git.


In [ ]:
RUN_ADD_DATA = False

if RUN_ADD_DATA:
    PriceFM_Plus.download_data("raw")
    PriceFM_Plus.download_data("processed")
    token = os.environ["ENTSOE_API_TOKEN"]
    updates = PriceFM_Plus.add_data("2026-10-01", "2026-10-08", token)
    print(updates)


## 7. Train and evaluate a custom model

`customize` trains, validates, tests, and prints region-level pinball loss, interval score, MAE, and RMSE. The six dates give training start/end, validation start/end, and test start/end. Each end is exclusive. The short, one-region, one-epoch example checks the workflow; it does **not** reproduce the paper. Omit a hyperparameter to use its paper setting. A full run can require a capable GPU and substantial time.


In [ ]:
RUN_CUSTOM_TRAINING = False

if RUN_CUSTOM_TRAINING:
    PriceFM_Plus.download_data("processed")
    result = PriceFM_Plus.customize(
        dates=[
            "2025-01-01", "2025-01-14",  # training
            "2025-01-14", "2025-01-16",  # validation
            "2025-01-16", "2025-01-18",  # test
        ],
        feature_case="Realistic",  # or "TSO", "Oracle"
        regions=["AT"],
        hyperparams={"epochs": 1, "microbatch_size": 64, "batch_size": 64},
    )
    print(result)


## What is and is not reproduced

The paper's table is recalculated from archived native-price forecasts. Figure PDFs are regenerated from released plotting inputs. Forecast inference reruns archived weights and may differ slightly across TensorFlow or hardware versions. Custom training creates a new experiment and is not a substitute for the archived paper weights. Fold 4 has no held-out test forecasts.
